In [ ]:
# Install + Upload dataset
!pip install -q albumentations optuna einops timm fvcore huggingface_hub

import os, glob, zipfile
from google.colab import files

ROOT = "/content/data/IDRiD"
os.makedirs(ROOT, exist_ok=True)

print("Select your dataset zip (B__Disease_Grading.zip) ...")
uploaded = files.upload()
for fname in uploaded.keys():
    with zipfile.ZipFile(fname, "r") as z:
        z.extractall(ROOT)
    os.remove(fname)
    print("Extracted:", fname)

# ---- check
csvs = glob.glob(f"{ROOT}/**/*.csv", recursive=True)
imgs = glob.glob(f"{ROOT}/**/*.jpg", recursive=True)
print("CSV files:", *csvs, sep="\n  ")
print("Images:", len(imgs))

Select your dataset zip (B__Disease_Grading.zip) ...


Saving B. Disease Grading.zip to B. Disease Grading.zip
Extracted: B. Disease Grading.zip
CSV files:
  /content/data/IDRiD/B. Disease Grading/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv
  /content/data/IDRiD/B. Disease Grading/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv
Images: 516


In [ ]:
#  Locate dataset folders
import os, glob

ROOT = "/content/data/IDRiD"
csv_train = glob.glob(f"{ROOT}/**/*Training Labels*.csv", recursive=True)[0]
csv_test  = glob.glob(f"{ROOT}/**/*Testing Labels*.csv",  recursive=True)[0]
img_train_dir = [d for d in glob.glob(f"{ROOT}/**/a. Training Set", recursive=True) if "Original Images" in d][0]
img_test_dir  = [d for d in glob.glob(f"{ROOT}/**/b. Testing Set",  recursive=True) if "Original Images" in d][0]

print("Train CSV :", csv_train)
print("Test CSV  :", csv_test)
print("Train imgs:", img_train_dir, len(os.listdir(img_train_dir)))   # 413
print("Test imgs :", img_test_dir,  len(os.listdir(img_test_dir)))    # 103

Train CSV : /content/data/IDRiD/B. Disease Grading/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv
Test CSV  : /content/data/IDRiD/B. Disease Grading/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv
Train imgs: /content/data/IDRiD/B. Disease Grading/1. Original Images/a. Training Set 413
Test imgs : /content/data/IDRiD/B. Disease Grading/1. Original Images/b. Testing Set 103


In [ ]:
# Configuration + seed
import random, numpy as np, torch

class CFG:
    img_size      = 224
    val_fraction  = 0.20
    split_seed    = 42
    seed          = 42
    retina_thr    = 10
    feat_dim      = 768
    n_dr, n_dme   = 5, 3
    lr            = 1e-4
    batch_size    = 32
    weight_decay  = 1e-4
    dropout       = 0.30
    epochs        = 100
    lambda_dr     = 1.0
    lambda_dme    = 1.0
    ln_eps        = 1e-6
    drop_path     = 0.2
    num_workers   = 2

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

set_seed(CFG.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

Device: cuda Tesla T4


Data Splitting

In [ ]:
#  Labels + data splitting
import pandas as pd, hashlib
from sklearn.model_selection import train_test_split

def read_labels(csv_path, img_dir, part):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    df = df[["Image name", "Retinopathy grade", "Risk of macular edema"]].dropna()
    df.columns = ["name", "dr", "dme"]
    df["name"] = df["name"].astype(str).str.strip()
    df[["dr", "dme"]] = df[["dr", "dme"]].astype(int)
    df["path"] = [glob.glob(os.path.join(img_dir, n + ".*"))[0] for n in df["name"]]
    df["part"] = part
    return df.reset_index(drop=True)

dev_df  = read_labels(csv_train, img_train_dir, "train")
test_df = read_labels(csv_test,  img_test_dir,  "test")

tr_idx, va_idx = train_test_split(np.arange(len(dev_df)), test_size=CFG.val_fraction,
                                  random_state=CFG.split_seed, shuffle=True)
train_df = dev_df.iloc[tr_idx].reset_index(drop=True)
val_df   = dev_df.iloc[va_idx].reset_index(drop=True)

def class_table(train, val, test):
    rows = {}
    for n, d in (("Training", train), ("Validation", val), ("Held-out Test", test)):
        r = {"Images": len(d)}
        r.update({f"DR {g}": int((d.dr == g).sum()) for g in range(5)})
        r.update({f"DME {g}": int((d.dme == g).sum()) for g in range(3)})
        rows[n] = r
    t = pd.DataFrame(rows).T; t.loc["Total"] = t.sum(); return t

class_dist = class_table(train_df, val_df, test_df)
print(class_dist.to_string())

md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
dev_hashes = {md5(p) for p in dev_df.path}
print("Duplicate images between development and test:", sum(md5(p) in dev_hashes for p in test_df.path))

os.makedirs("/content/outputs", exist_ok=True)
for n, d in (("train", train_df), ("val", val_df), ("test", test_df)):
    d[["name", "dr", "dme"]].to_csv(f"/content/outputs/split_{n}.csv", index=False)
class_dist.to_csv("/content/outputs/class_distribution.csv")

                    Images   DR 0   DR 1   DR 2   DR 3   DR 4   DME 0  DME 1  DME 2
Training               330    111     13    108     58     40     144     32    154
Validation              83     23      7     28     16      9      33      9     41
Held-out Test          103     34      5     32     19     13      45     10     48
Total                  516    168     25    168     93     62     222     51    243

Duplicate images between development and test: 0


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json, hashlib
ART = "/content/drive/MyDrive/DR_DME_outputs/artifacts"
os.makedirs(ART, exist_ok=True)

sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()

split = pd.concat([
    dev_df.iloc[tr_idx].assign(split="train", dev_row=tr_idx),
    dev_df.iloc[va_idx].assign(split="val",   dev_row=va_idx),
    test_df.assign(split="test", dev_row=-1),
])
split["file_sha256"] = [sha(p) for p in split.path]
split[["name","split","dev_row","dr","dme","file_sha256"]].to_csv(f"{ART}/split_assignment.csv", index=False)

json.dump({"split_seed": CFG.split_seed, "val_fraction": CFG.val_fraction,
           "train_idx": tr_idx.tolist(), "val_idx": va_idx.tolist(),
           "n_train": len(tr_idx), "n_val": len(va_idx), "n_test": len(test_df)},
          open(f"{ART}/split_indices.json", "w"), indent=1)

Preprocessing

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def retina_bbox(rgb, thr=CFG.retina_thr):
    mask = (rgb.max(axis=2) > thr).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        h, w = mask.shape
        return 0, 0, w, h
    return cv2.boundingRect(max(cnts, key=cv2.contourArea))


def preprocess_steps(path, size=CFG.img_size):
    """Return (original, cropped, resized) for one image."""
    bgr = cv2.imread(str(path))
    if bgr is None:
        raise FileNotFoundError(f"Could not read: {path}")
    raw = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    x, y, w, h = retina_bbox(raw)
    crop = raw[y:y + h, x:x + w]
    resized = cv2.resize(crop, (size, size), interpolation=cv2.INTER_AREA)
    return raw, crop, resized

Augmentation

In [ ]:
#  Augmentation + Dataset + Loaders

import albumentations as A
import torch
import numpy as np
import random
import cv2
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader

train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),

    A.Rotate(
        limit=10,
        border_mode=cv2.BORDER_CONSTANT,
        p=0.5
    ),

    A.Affine(
        scale=(0.9, 1.1),
        translate_percent={
            "x": (-0.05, 0.05),
            "y": (-0.05, 0.05)
        },
        shear=(-3, 3),
        p=0.4
    ),

    A.RandomBrightnessContrast(
        brightness_limit=0.1,
        contrast_limit=0.1,
        p=0.4
    ),
])

# Helper wrapper to map to your defined preprocess_steps function
def load_and_preprocess(path):

    _, _, resized = preprocess_steps(path, size=CFG.img_size)
    return resized

class IDRiDGrading(Dataset):

    def __init__(self, df, aug=None):
        self.df = df.reset_index(drop=True)
        self.aug = aug

        self.cache = [
            load_and_preprocess(p)
            for p in self.df.path
        ]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):

        img = self.cache[i]

        if self.aug is not None:
            img = self.aug(image=img)["image"]

        x = torch.from_numpy(
            (img.astype(np.float32) / 255.0)
            .transpose(2, 0, 1)
            .copy()
        )

        return (
            x,
            int(self.df.dr[i]),
            int(self.df.dme[i]),
            self.df.name[i]
        )


def seed_worker(wid):
    s = torch.initial_seed() % 2**32
    np.random.seed(s)
    random.seed(s)


train_ds = IDRiDGrading(train_df, aug=train_aug)
val_ds   = IDRiDGrading(val_df)
test_ds  = IDRiDGrading(test_df)


g = torch.Generator()
g.manual_seed(CFG.seed)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False,
    num_workers=CFG.num_workers
)

test_loader = DataLoader(
    test_ds,
    batch_size=64,
    shuffle=False,
    num_workers=CFG.num_workers
)

In [ ]:
import sys, platform, subprocess, importlib.metadata as md
pk = ["torch","torchvision","timm","numpy","pandas","scikit-learn","scipy","albumentations",
      "opencv-python","optuna","einops","huggingface_hub","matplotlib"]
env = {"python": sys.version, "platform": platform.platform(),
       "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       "packages": {p: md.version(p) for p in pk if md.distributions()}}
json.dump(env, open(f"{ART}/environment.json","w"), indent=1)
open(f"{ART}/requirements_frozen.txt","w").write(
    subprocess.run([sys.executable,"-m","pip","freeze"],capture_output=True,text=True).stdout)

json.dump({"CFG": {k:v for k,v in vars(CFG).items() if not k.startswith("_")},
           "preprocess": {"retina_thr": CFG.retina_thr, "img_size": CFG.img_size,
                          "resize": "INTER_AREA", "normalisation": "none (0-1 scaling)"},
           "augmentation": str(train_aug)},
          open(f"{ART}/config.json","w"), indent=1, default=str)

In [ ]:
# Download ImageNet-pretrained EfficientVMamba-S weights
import os, glob, warnings, logging

# silence the Hugging Face "unauthenticated requests" warning and progress bars
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
warnings.filterwarnings("ignore")
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

from huggingface_hub import snapshot_download

WEIGHT_DIR = "/content/weights/evm_small"
ckpts = glob.glob(f"{WEIGHT_DIR}/**/*.ckpt", recursive=True)
if not ckpts:                                            # download only once
    snapshot_download("TerryPei/EfficientVMamba-Small", local_dir=WEIGHT_DIR)
    ckpts = glob.glob(f"{WEIGHT_DIR}/**/*.ckpt", recursive=True)

BACKBONE_CKPT = ckpts[0] if ckpts else None
if BACKBONE_CKPT:
    print(f"✅ ImageNet checkpoint ready: {BACKBONE_CKPT} ({os.path.getsize(BACKBONE_CKPT) / 1e6:.1f} MB)")
else:
    print("❌ Checkpoint not found - check the internet connection and re-run this cell")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

✅ ImageNet checkpoint ready: /content/weights/evm_small/efficient_vmamba_small.ckpt (45.3 MB)


In [ ]:
# EfficientVMamba-S backbone (no CUDA kernel needed)
import math, torch.nn as nn, torch.nn.functional as F
import torch.utils.checkpoint as cp
from collections import OrderedDict

USE_CHECKPOINT = True     # saves GPU memory during training (recomputes EVSS blocks in backward)

def selective_scan(u, delta, A, B, C, D, delta_bias, softplus=True):
    """Pure-PyTorch selective scan.
    u, delta: (b, K*d, L) | A: (K*d, N) | B, C: (b, K, N, L) | D: (K*d,)"""
    with torch.autocast(device_type="cuda", enabled=False):
        u, delta, A, B, C, D, delta_bias = [t.float() for t in (u, delta, A, B, C, D, delta_bias)]
        b, KD, L = u.shape; K, N = B.shape[1], A.shape[1]; d = KD // K
        delta = delta + delta_bias[None, :, None]
        if softplus:
            delta = F.softplus(delta)
        Bt = B.permute(0, 1, 3, 2).unsqueeze(2)                       # b,K,1,L,N
        Ct = C.permute(0, 1, 3, 2)                                    # b,K,L,N
        dA  = torch.exp(delta.view(b, K, d, L, 1) * A.view(1, K, d, 1, N))   # b,K,d,L,N
        dBu = (delta * u).view(b, K, d, L, 1) * Bt                           # b,K,d,L,N
        h = u.new_zeros(b, K, d, N); hs = []
        for t in range(L):                                            # h_t = dA_t * h_{t-1} + dB_t u_t
            h = torch.addcmul(dBu[:, :, :, t], dA[:, :, :, t], h)
            hs.append(h)
        hs = torch.stack(hs, 3)                                       # b,K,d,L,N
        y = torch.einsum("bkdln,bkln->bkdl", hs, Ct).reshape(b, KD, L)
        return y + u * D[None, :, None]

print("Selective-scan backend: pure PyTorch (vectorised) + gradient checkpointing")


def efficient_scan(x, s=2):
    B, C, H, W = x.shape
    if H % s or W % s: x = F.pad(x, (0, (-W) % s, 0, (-H) % s))
    xt = x.transpose(2, 3)
    return torch.stack([x[:, :, ::s, ::s].reshape(B, C, -1), xt[:, :, ::s, 1::s].reshape(B, C, -1),
                        x[:, :, ::s, 1::s].reshape(B, C, -1), xt[:, :, 1::s, 1::s].reshape(B, C, -1)], 1)

def efficient_merge(ys, H0, W0, s=2):
    B, K, C, L = ys.shape; H, W = math.ceil(H0 / s), math.ceil(W0 / s)
    y = ys.new_zeros(B, C, H * s, W * s)
    y[:, :, ::s, ::s]   = ys[:, 0].reshape(B, C, H, W)
    y[:, :, 1::s, ::s]  = ys[:, 1].reshape(B, C, W, H).transpose(2, 3)
    y[:, :, ::s, 1::s]  = ys[:, 2].reshape(B, C, H, W)
    y[:, :, 1::s, 1::s] = ys[:, 3].reshape(B, C, W, H).transpose(2, 3)
    return y[:, :, :H0, :W0].reshape(B, C, -1)

class Permute(nn.Module):
    def __init__(self, *d): super().__init__(); self.d = d
    def forward(self, x): return x.permute(*self.d)

class DropPath(nn.Module):
    def __init__(self, p=0.): super().__init__(); self.p = p
    def forward(self, x):
        if not self.training or self.p == 0: return x
        m = x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(1 - self.p)
        return x * m / (1 - self.p)

class SS2D(nn.Module):
    def __init__(self, d_model, d_state=16, ssm_ratio=2.0, step=2):
        super().__init__()
        di = int(ssm_ratio * d_model); R = math.ceil(d_model / 16); N, K = d_state, 4
        self.R, self.N, self.K, self.step = R, N, K, step
        self.in_proj = nn.Linear(d_model, 2 * di, bias=False); self.act = nn.SiLU()
        self.conv2d = nn.Conv2d(di, di, 3, padding=1, groups=di)
        xw = torch.empty(K, R + 2 * N, di)
        for k in range(K): nn.init.kaiming_uniform_(xw[k], a=math.sqrt(5))
        self.x_proj_weight = nn.Parameter(xw)
        self.dt_projs_weight = nn.Parameter(torch.empty(K, di, R).uniform_(-R ** -0.5, R ** -0.5))
        dt = torch.exp(torch.rand(K, di) * (math.log(0.1) - math.log(0.001)) + math.log(0.001)).clamp(min=1e-4)
        self.dt_projs_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))
        self.A_logs = nn.Parameter(torch.log(torch.arange(1, N + 1, dtype=torch.float32).repeat(K * di, 1)))
        self.Ds = nn.Parameter(torch.ones(K * di))
        self.out_norm = nn.LayerNorm(di); self.out_proj = nn.Linear(di, d_model, bias=False)
    def forward(self, x):
        B, H, W, _ = x.shape
        x, z = self.in_proj(x).chunk(2, -1); z = self.act(z)
        x = self.act(self.conv2d(x.permute(0, 3, 1, 2).contiguous())); D = x.shape[1]
        xs = efficient_scan(x, self.step); L = xs.shape[-1]
        dts, Bs, Cs = torch.split(torch.einsum("bkdl,kcd->bkcl", xs, self.x_proj_weight), [self.R, self.N, self.N], 2)
        dts = torch.einsum("bkrl,kdr->bkdl", dts, self.dt_projs_weight)
        ys = selective_scan(xs.reshape(B, -1, L), dts.reshape(B, -1, L), -torch.exp(self.A_logs.float()),
                            Bs, Cs, self.Ds, self.dt_projs_bias.reshape(-1), True).reshape(B, 4, D, L)
        y = self.out_norm(efficient_merge(ys, H, W, self.step).transpose(1, 2)).reshape(B, H, W, D).to(z.dtype)
        return self.out_proj(y * z)

class BiAttn(nn.Module):
    def __init__(self, c, r=0.125):
        super().__init__()
        self.norm = nn.LayerNorm(c); self.global_reduce = nn.Linear(c, int(c * r))
        self.act_fn = nn.GELU(); self.channel_select = nn.Linear(int(c * r), c)
    def forward(self, x):
        g = self.norm(x).mean([1, 2], keepdim=True)
        return x * torch.sigmoid(self.channel_select(self.act_fn(self.global_reduce(g))))

class EVSSBlock(nn.Module):
    def __init__(self, dim, dp=0.):
        super().__init__()
        self.norm = nn.LayerNorm(dim); self.op = SS2D(dim)
        self.conv_branch = nn.Sequential(nn.Conv2d(dim, dim, 3, padding=1, groups=dim), nn.BatchNorm2d(dim),
                                         nn.GELU(), nn.Conv2d(dim, dim, 1))
        self.se = BiAttn(dim); self.drop_path = DropPath(dp)
    def _forward(self, inp):
        x = self.norm(inp)
        xc = self.conv_branch(x.permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
        return inp + self.drop_path(self.se(self.op(x)) + self.se(xc))
    def forward(self, inp):
        if USE_CHECKPOINT and self.training and torch.is_grad_enabled():
            return cp.checkpoint(self._forward, inp, use_reentrant=False)
        return self._forward(inp)

class SqueezeExcite(nn.Module):
    def __init__(self, c, r):
        super().__init__()
        self.avgp = nn.AdaptiveAvgPool2d(1); self.conv_reduce = nn.Conv2d(c, r, 1)
        self.act_fn = nn.ReLU(inplace=True); self.conv_expand = nn.Conv2d(r, c, 1)
    def forward(self, x): return x * torch.sigmoid(self.conv_expand(self.act_fn(self.conv_reduce(self.avgp(x)))))

class InvertedResidual(nn.Module):
    def __init__(self, c, e=4, se=0.125, dp=0.):
        super().__init__(); m = c * e
        self.conv_pw  = nn.Sequential(nn.Conv2d(c, m, 1, bias=False), nn.BatchNorm2d(m), nn.ReLU(inplace=True))
        self.conv_dw  = nn.Sequential(nn.Conv2d(m, m, 3, 1, 1, groups=m, bias=False), nn.BatchNorm2d(m), nn.ReLU(inplace=True))
        self.se       = SqueezeExcite(m, int(m * se))
        self.conv_pw2 = nn.Sequential(nn.Conv2d(m, c, 1, bias=False), nn.BatchNorm2d(c))
        self.drop = DropPath(dp)
    def forward(self, x): return x + self.drop(self.conv_pw2(self.se(self.conv_dw(self.conv_pw(x)))))

class PatchMerging2D(nn.Module):
    def __init__(self, d, od):
        super().__init__(); self.reduction = nn.Linear(4 * d, od, bias=False); self.norm = nn.LayerNorm(4 * d)
    def forward(self, x):
        x = torch.cat([x[:, 0::2, 0::2], x[:, 1::2, 0::2], x[:, 0::2, 1::2], x[:, 1::2, 1::2]], -1)
        return self.reduction(self.norm(x))

class EfficientVMambaS(nn.Module):
    """stem -> 2xEVSS@56x56x96 -> 2xEVSS@28x28x192 -> 4xInRes@14x14x384 -> 2xInRes@7x7x768"""
    def __init__(self, depths=(2, 2, 4, 2), dims=(96, 192, 384, 768), drop_path=0.2):
        super().__init__()
        self.patch_embed = nn.Sequential(nn.Conv2d(3, dims[0], 4, 4), Permute(0, 2, 3, 1), nn.LayerNorm(dims[0]))
        dpr = torch.linspace(0, drop_path, sum(depths)).tolist(); self.layers = nn.ModuleList()
        for i, dep in enumerate(depths):
            dp = dpr[sum(depths[:i]):sum(depths[:i + 1])]
            down = PatchMerging2D(dims[i], dims[i + 1]) if i < 3 else nn.Identity()
            if i < 2:
                self.layers.append(nn.Sequential(OrderedDict(
                    blocks=nn.Sequential(*[EVSSBlock(dims[i], dp[j]) for j in range(dep)]), downsample=down)))
            else:
                self.layers.append(nn.Sequential(Permute(0, 3, 1, 2),
                                                 *[InvertedResidual(dims[i], dp=dp[j]) for j in range(dep)],
                                                 Permute(0, 2, 3, 1), down))
        self.classifier = nn.Sequential(OrderedDict(norm=nn.LayerNorm(dims[-1])))
    def forward(self, x):
        x = self.patch_embed(x)
        for l in self.layers: x = l(x)
        return self.classifier.norm(x).permute(0, 3, 1, 2).contiguous()     # B,768,7,7

def load_imagenet_weights(model, path):
    if not path or not os.path.exists(path):
        print("WARNING: no ImageNet weights -> random init"); return model
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = ck.get("model", ck.get("state_dict", ck))
    sd = {k.replace("module.", ""): v for k, v in sd.items()}
    own = model.state_dict()
    sd = {k: v for k, v in sd.items() if k in own and own[k].shape == v.shape}
    missing = model.load_state_dict(sd, strict=False).missing_keys
    return model

# ---- quick test: shape, memory and speed of one training step
import time
bb = load_imagenet_weights(EfficientVMambaS(drop_path=CFG.drop_path), BACKBONE_CKPT).to(device).train()
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
x = torch.rand(CFG.batch_size, 3, 224, 224, device=device)
t0 = time.time()
with torch.autocast(device.type, dtype=torch.float16):
    f = bb(x)
f.float().mean().backward()
if device.type == "cuda":
    torch.cuda.synchronize()
print("Backbone output:", tuple(f.shape))                                    # (32, 768, 7, 7)
mem_str = f" | peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB" if device.type == "cuda" else ""
print(f"1 train step (batch {CFG.batch_size}): {time.time() - t0:.1f}s{mem_str}")
print(f"Backbone params: {sum(p.numel() for p in bb.parameters()) / 1e6:.2f} M")
del bb, x, f
if device.type == "cuda":
    torch.cuda.empty_cache()

Selective-scan backend: pure PyTorch (vectorised) + gradient checkpointing
Backbone output: (32, 768, 7, 7)
1 train step (batch 32): 74.3s | peak GPU memory 8.6 GB
Backbone params: 10.2 M


In [ ]:
# Proposed multi-task model
class TokenAttention(nn.Module):
    """Single-head attention, pre-LayerNorm, Q/K/V 768x768 with bias."""
    def __init__(self, d=768, eps=1e-6):
        super().__init__()
        self.ln_q, self.ln_kv = nn.LayerNorm(d, eps=eps), nn.LayerNorm(d, eps=eps)
        self.q, self.k, self.v = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.scale = 1 / math.sqrt(d)
    def forward(self, q_src, kv_src):
        q = self.q(self.ln_q(q_src)); kv = self.ln_kv(kv_src)
        w = torch.softmax(q @ self.k(kv).transpose(-2, -1) * self.scale, -1)   # B,49,49
        return w @ self.v(kv), w

class CoralHead(nn.Module):
    """p_k = sigmoid(w^T h + b_k): shared w, 4 threshold biases."""
    def __init__(self, d, n=5):
        super().__init__(); self.fc = nn.Linear(d, 1, bias=False); self.bias = nn.Parameter(torch.zeros(n - 1))
    def forward(self, h): return self.fc(h) + self.bias

class MultiTaskDRDME(nn.Module):
    def __init__(self, cfg=CFG, ckpt=None):
        super().__init__()
        D = cfg.feat_dim
        self.backbone = load_imagenet_weights(EfficientVMambaS(drop_path=cfg.drop_path), ckpt)
        self.proj_dr, self.proj_dme = nn.Linear(D, D), nn.Linear(D, D)
        self.att_dr  = TokenAttention(D, cfg.ln_eps)                          # DR  <- DME
        self.att_dme = TokenAttention(D, cfg.ln_eps)                          # DME <- DR
        self.drop = nn.Dropout(cfg.dropout)
        self.head_dr  = CoralHead(D, cfg.n_dr)
        self.head_dme = nn.Linear(D, cfg.n_dme)
        for n, m in self.named_modules():                                     # Xavier for new layers
            if not n.startswith("backbone") and isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x, return_all=False):
        fmap = self.backbone(x)                                   # B,768,7,7
        Fs = fmap.flatten(2).transpose(1, 2)                      # B,49,768
        F_dr, F_dme = self.proj_dr(Fs), self.proj_dme(Fs)
        a_dr,  w_dr  = self.att_dr(F_dr, F_dme)
        a_dme, w_dme = self.att_dme(F_dme, F_dr)
        Fh_dr, Fh_dme = F_dr + a_dr, F_dme + a_dme
        out = {"dr_logits":  self.head_dr(self.drop(Fh_dr.mean(1))),
               "dme_logits": self.head_dme(self.drop(Fh_dme.mean(1)))}
        if return_all:
            out.update(fmap=fmap, tokens_dr=Fh_dr, tokens_dme=Fh_dme, attn_dr=w_dr, attn_dme=w_dme)
        return out

def coral_levels(y, n=5):
    return (y[:, None] > torch.arange(n - 1, device=y.device)[None]).float()

def dr_decode(logits):
    p = torch.sigmoid(logits)
    pred = (p > 0.5).sum(1)
    cum = torch.cat([torch.ones_like(p[:, :1]), p, torch.zeros_like(p[:, :1])], 1)
    probs = (cum[:, :-1] - cum[:, 1:]).clamp_min(0)
    return pred, probs / probs.sum(1, keepdim=True).clamp_min(1e-12)

def multitask_loss(out, y_dr, y_dme, cfg=CFG):
    l_dr = F.binary_cross_entropy_with_logits(out["dr_logits"].float(), coral_levels(y_dr),
                                              reduction="none").sum(1).mean()
    l_dme = F.cross_entropy(out["dme_logits"].float(), y_dme)
    return cfg.lambda_dr * l_dr + cfg.lambda_dme * l_dme, l_dr.item(), l_dme.item()

# ---- sanity check on one real batch
model = MultiTaskDRDME(CFG, BACKBONE_CKPT).to(device)
x, ydr, ydme, _ = next(iter(train_loader))
x, ydr, ydme = x.to(device), ydr.to(device), ydme.to(device)
out = model(x, return_all=True)
loss, ldr, ldme = multitask_loss(out, ydr, ydme)
loss.backward()
print({k: tuple(v.shape) for k, v in out.items()})
print(f"loss {loss.item():.4f} (DR {ldr:.4f}, DME {ldme:.4f})")
print(f"Total params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")
model.zero_grad(set_to_none=True)

{'dr_logits': (32, 4), 'dme_logits': (32, 3), 'fmap': (32, 768, 7, 7), 'tokens_dr': (32, 49, 768), 'tokens_dme': (32, 49, 768), 'attn_dr': (32, 49, 49), 'attn_dme': (32, 49, 49)}
loss 5.2379 (DR 3.5683, DME 1.6695)
Total params: 14.9 M


In [ ]:
# Configurable version of the multi-task model for the architectural ablation
class FlexMultiTask(nn.Module):
    """proj: task-specific projections | interaction: none, bi, dr<-dme, dme<-dr | dr_head: coral or softmax"""
    def __init__(self, cfg=CFG, ckpt=None, dropout=None, proj=True, interaction="bi", dr_head="coral"):
        super().__init__()
        D = cfg.feat_dim
        self.interaction, self.dr_head_type = interaction, dr_head
        self.backbone = load_imagenet_weights(EfficientVMambaS(drop_path=cfg.drop_path), ckpt)
        self.proj_dr  = nn.Linear(D, D) if proj else nn.Identity()
        self.proj_dme = nn.Linear(D, D) if proj else nn.Identity()
        self.att_dr  = TokenAttention(D, cfg.ln_eps) if interaction in ("bi", "dr<-dme") else None
        self.att_dme = TokenAttention(D, cfg.ln_eps) if interaction in ("bi", "dme<-dr") else None
        self.drop = nn.Dropout(cfg.dropout if dropout is None else dropout)
        self.head_dr  = CoralHead(D, cfg.n_dr) if dr_head == "coral" else nn.Linear(D, cfg.n_dr)
        self.head_dme = nn.Linear(D, cfg.n_dme)
        for n, m in self.named_modules():
            if not n.startswith("backbone") and isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        Fs = self.backbone(x).flatten(2).transpose(1, 2)          # B,49,768
        F_dr, F_dme = self.proj_dr(Fs), self.proj_dme(Fs)
        Fh_dr  = F_dr  + self.att_dr(F_dr, F_dme)[0]  if self.att_dr  is not None else F_dr
        Fh_dme = F_dme + self.att_dme(F_dme, F_dr)[0] if self.att_dme is not None else F_dme
        return {"dr_logits":  self.head_dr(self.drop(Fh_dr.mean(1))),
                "dme_logits": self.head_dme(self.drop(Fh_dme.mean(1)))}

In [ ]:
# Imports and flags needed by the training engine
import copy, time, json
from sklearn.metrics import f1_score, cohen_kappa_score, matthews_corrcoef

USE_AMP = device.type == "cuda"

In [ ]:
# LOSS, METRICS, TRAINING ENGINE
def coral_levels(y, n=5):
    return (y[:, None] > torch.arange(n - 1, device=y.device)[None]).float()

def decode_dr(logits, head_type):
    return (torch.sigmoid(logits) > 0.5).sum(1) if head_type == "coral" else logits.argmax(1)

def compute_loss(out, ydr, ydme, dr_type, lam):
    loss = 0.0
    if out["dr_logits"] is not None:
        if dr_type == "coral":
            l = F.binary_cross_entropy_with_logits(out["dr_logits"].float(), coral_levels(ydr), reduction="none").sum(1).mean()
        else:
            l = F.cross_entropy(out["dr_logits"].float(), ydr)
        loss = loss + lam[0] * l
    if out["dme_logits"] is not None:
        loss = loss + lam[1] * F.cross_entropy(out["dme_logits"].float(), ydme)
    return loss

def grade_metrics(y, p, n, full=True):
    m = {"f1": float(f1_score(y, p, labels=list(range(n)), average="macro", zero_division=0)),
         "acc": float((y == p).mean())}
    if full:
        m.update(kappa=float(cohen_kappa_score(y, p)), mcc=float(matthews_corrcoef(y, p)),
                 recall=[float(((p == k) & (y == k)).sum() / max(1, (y == k).sum())) for k in range(n)],
                 correct=int((y == p).sum()))
    return m

@torch.no_grad()
def predict(model, loader):
    model.eval(); R = {k: [] for k in ("y_dr", "y_dme", "p_dr", "p_dme")}
    for x, ydr, ydme, _ in loader:
        with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
            out = model(x.to(device))
        R["y_dr"].append(ydr); R["y_dme"].append(ydme)
        if out["dr_logits"] is not None:  R["p_dr"].append(decode_dr(out["dr_logits"].float(), model.dr_head_type).cpu())
        if out["dme_logits"] is not None: R["p_dme"].append(out["dme_logits"].argmax(1).cpu())
    return {k: (torch.cat(v).numpy() if len(v) else None) for k, v in R.items()}

def score(R, alpha=0.5, full=False):
    s = {"dr": None, "dme": None}
    if R["p_dr"]  is not None: s["dr"]  = grade_metrics(R["y_dr"],  R["p_dr"],  5, full)
    if R["p_dme"] is not None: s["dme"] = grade_metrics(R["y_dme"], R["p_dme"], 3, full)
    if s["dr"] and s["dme"]: s["J"] = alpha * (1 - s["dr"]["f1"]) + (1 - alpha) * (1 - s["dme"]["f1"])
    else:                    s["J"] = 1 - (s["dr"] or s["dme"])["f1"]
    return s

def param_groups(model, wd):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if p.ndim <= 1 or "bias" in n or "norm" in n or "ln_" in n or "A_logs" in n or n.endswith("Ds"): no_decay.append(p)
        else: decay.append(p)
    return [{"params": decay, "weight_decay": wd}, {"params": no_decay, "weight_decay": 0.0}]

def warmup_cosine(opt, total_steps, base_lr, warmup_frac=0.05, min_lr=1e-6):
    warm = max(1, math.ceil(warmup_frac * total_steps))
    def f(step):
        if step < warm: return (step + 1) / warm
        prog = min(1.0, (step - warm) / max(1, total_steps - warm))
        return (min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * prog))) / base_lr
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def train_eval(model_fn, D, hp, seed=42, lam=(1.0, 1.0), eval_test=True):
    set_seed(seed)
    model = model_fn(hp["dropout"]).to(device)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    gen = torch.Generator(); gen.manual_seed(seed)
    tl = DataLoader(D.train_ds, batch_size=int(hp["batch_size"]), shuffle=True, num_workers=CFG.num_workers,
                    worker_init_fn=seed_worker, generator=gen, pin_memory=device.type == "cuda")
    vl = DataLoader(D.val_ds, batch_size=64, shuffle=False, num_workers=CFG.num_workers)
    epochs = int(hp["epochs"])
    opt = torch.optim.AdamW(param_groups(model, hp["weight_decay"]), lr=hp["lr"], betas=(0.9, 0.999), eps=1e-8)
    sched = warmup_cosine(opt, epochs * len(tl), hp["lr"])
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best, best_state, t0, ep_times = {"J": float("inf"), "epoch": 0}, None, time.time(), []
    for ep in range(1, epochs + 1):
        model.train(); te = time.time()
        for x, ydr, ydme, _ in tl:
            x, ydr, ydme = x.to(device, non_blocking=True), ydr.to(device), ydme.to(device)
            with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
                out = model(x)
            loss = compute_loss(out, ydr, ydme, model.dr_head_type, lam)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
        ep_times.append(time.time() - te)
        sc = score(predict(model, vl))
        if sc["J"] < best["J"]:
            best = {"J": sc["J"], "epoch": ep}
            if eval_test: best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
    res = {"hp": hp, "params_M": n_params, "best_J": best["J"], "best_epoch": best["epoch"],
           "seconds": time.time() - t0, "epoch_time_s": float(np.mean(ep_times)), "val": None, "test": None, "pred_test": None}
    if eval_test:
        model.load_state_dict(best_state)
        res["val"] = score(predict(model, vl), full=True)
        tlo = DataLoader(D.test_ds, batch_size=64, shuffle=False, num_workers=CFG.num_workers)
        R = predict(model, tlo); res["test"] = score(R, full=True)
        res["pred_test"] = {k: (v.tolist() if v is not None else None) for k, v in R.items()}
    del model, opt; torch.cuda.empty_cache()
    return res

def pct(d, task, key):
    return "–" if (d is None or d.get(task) is None) else f"{100 * d[task][key]:.2f}"

def table_row(name, res):
    v, t = res["val"], res["test"]
    return {"Configuration": name, "Val. DR F1 (%)": pct(v, "dr", "f1"), "Val. DME F1 (%)": pct(v, "dme", "f1"),
            "Val. J": f"{v['J']:.4f}", "Test DR Acc (%)": pct(t, "dr", "acc"), "Test DR Kappa (%)": pct(t, "dr", "kappa"),
            "Test DR MCC (%)": pct(t, "dr", "mcc"), "Test DME Acc (%)": pct(t, "dme", "acc"),
            "Test DME Kappa (%)": pct(t, "dme", "kappa"), "Test DME MCC (%)": pct(t, "dme", "mcc")}

def cached(path, fn):
    """Run fn() once and store the result as JSON, so an interrupted run can resume."""
    if os.path.exists(path): return json.load(open(path))
    r = fn(); json.dump(r, open(path, "w"), indent=1); return r

def init_env():
    resolve_backbone_ckpt(); set_seed(CFG.seed)

def proposed_fn(dropout, **kw):
    return FlexMultiTask(CFG, BACKBONE_CKPT, dropout=dropout, **kw)



In [ ]:
# Settings, data bundle and checkpoint helper used by the ablation run
from types import SimpleNamespace

FINAL_HP = dict(lr=1e-4, batch_size=32, weight_decay=1e-4, dropout=0.30, epochs=100)
OUT_DIR = "/content/drive/MyDrive/DR_DME_outputs/ablation"
os.makedirs(OUT_DIR, exist_ok=True)

def build_data():
    return SimpleNamespace(train_ds=train_ds, val_ds=val_ds, test_ds=test_ds)

def resolve_backbone_ckpt():
    global BACKBONE_CKPT
    found = glob.glob(f"{WEIGHT_DIR}/**/*.ckpt", recursive=True)
    if not found:
        snapshot_download("TerryPei/EfficientVMamba-Small", local_dir=WEIGHT_DIR)
        found = glob.glob(f"{WEIGHT_DIR}/**/*.ckpt", recursive=True)
    BACKBONE_CKPT = found[0] if found else None

In [ ]:
# ablation of architectural components
CONFIGS = [
    ("Shared backbone with dual heads",    dict(proj=False, interaction="none")),
    ("Without task-specific projections",  dict(proj=False, interaction="bi")),
    ("Without cross-task attention",       dict(proj=True,  interaction="none")),
    ("Unidirectional (DR <- DME)",         dict(proj=True,  interaction="dr<-dme")),
    ("Unidirectional (DME <- DR)",         dict(proj=True,  interaction="dme<-dr")),
    ("Categorical DR head (softmax)",      dict(proj=True,  interaction="bi", dr_head="softmax")),
    ("Proposed framework",                 dict(proj=True,  interaction="bi")),
]

def main():
    init_env(); D = build_data(); rows = []
    for i, (name, kw) in enumerate(CONFIGS):
        res = cached(os.path.join(OUT_DIR, f"abl_arch_{i}.json"),
                     lambda: train_eval(lambda d, kw=kw: proposed_fn(d, **kw), D, FINAL_HP, seed=42))
        rows.append(table_row(name, res))
    abl_table = pd.DataFrame(rows)
    abl_table.to_csv(os.path.join(OUT_DIR, "table_architecture_ablation.csv"), index=False)
    print("Ablation of architectural components\n", abl_table.to_string(index=False))

main()


Ablation of architectural components

Configuration                              Val. DR F1 (%)  Val. DME F1 (%)    Val. J  Test DR Acc (%)  Test DR Kappa (%)  Test DR MCC (%)  Test DME Acc (%)  Test DME Kappa (%)  Test DME MCC (%)
Shared backbone with dual heads                     94.04            93.41    0.0628            96.12              94.75            94.77             97.09               94.97             95.02
Without task-specific projections                   95.21            95.48    0.0465            96.12              95.02            95.10             97.09               95.18             95.25
Without cross-task attention                        96.16            95.48    0.0418            97.09              95.90            95.97             98.06               96.52             96.60
Unidirectional (DR <- DME)                          96.11            93.41    0.0524            97.09              96.17            96.24             97.09               95.30          